# Top2Vec: Temporal Topic Analysis

Analyze topic evolution over time (2000–2025) using **existing best Top2Vec models**
trained on all documents. No retraining needed.

Approach inspired by BERTopic's `topics_over_time()`:
1. Use the single trained model's topic assignments (`doc_top`)
2. Group documents by year using `submitted_date`
3. Compute per-year topic prevalence, coherence, and word evolution

**Advantage over sliced modeling:** Topics are consistent across all years
(same topic IDs), no alignment needed.

In [1]:
import gc
import time
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from top2vec import Top2Vec
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

## Configuration

In [2]:
LIST_SUBJECT = ["cs", "math", "physics"]

BASE_DIR = Path("../../../../data/preprocess")
MODEL_DIR = Path("../../../../models/top2vec/tuning")
RESULT_DIR = Path("../../../../results/top2vec/temporal")
VERSION = "v1"

# IRBO config
TOP_N_WORDS = 10
RBO_P = 0.9

# Create output directories
for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

print(f"Subjects: {LIST_SUBJECT}")
print(f"Model directory: {MODEL_DIR}")
print(f"Results directory: {RESULT_DIR}")

Subjects: ['cs', 'math', 'physics']
Model directory: ../../../../models/top2vec/tuning
Results directory: ../../../../results/top2vec/temporal


## Helper Functions

In [3]:
def load_dataset(subject: str) -> pd.DataFrame:
    """Load dataset with year column."""
    file_path = BASE_DIR / subject / "emb" / f"{VERSION}.csv"
    df = pd.read_csv(file_path)
    df["submitted_date"] = pd.to_datetime(df["submitted_date"])
    df["year"] = df["submitted_date"].dt.year
    return df


def compute_ctfidf_per_year(
    df, topic_col="topic", text_col="text", top_n=10,
    global_tuning=True, evolution_tuning=True,
):
    """
    Compute c-TF-IDF per topic per year with optional tuning.
    
    Mirrors BERTopic's topics_over_time() tuning approach:
      - global_tuning: average each (year, topic) c-TF-IDF with the
        global (all-year) topic c-TF-IDF to anchor representations.
      - evolution_tuning: average each (year, topic) c-TF-IDF with
        the previous year (t-1) to smooth transitions.
    
    Returns:
        topic_words_per_year: dict of {(year, topic_id): [word1, word2, ...]}
    """
    years = sorted(df["year"].unique())
    topics = sorted(df[topic_col].unique())
    
    # Group documents by (year, topic) and concatenate
    groups = df.groupby(["year", topic_col])[text_col].apply(
        lambda x: " ".join(x)
    ).reset_index()
    groups.columns = ["year", "topic", "text"]
    
    # Build vocabulary across all groups
    vectorizer = CountVectorizer(stop_words="english")
    tf_matrix = vectorizer.fit_transform(groups["text"])
    vocab = vectorizer.get_feature_names_out()
    
    # Compute IDF: log(N / df_t) where N = number of groups, df_t = groups containing term
    n_groups = tf_matrix.shape[0]
    df_t = (tf_matrix > 0).sum(axis=0).A1  # document frequency per term
    idf = np.log((n_groups + 1) / (df_t + 1)) + 1  # smoothed IDF
    
    # TF-IDF per group
    tfidf_matrix = tf_matrix.multiply(idf).toarray()
    
    # --- Global c-TF-IDF (topic only, ignoring year) ---
    global_tfidf = None
    global_topic_to_idx = {}
    if global_tuning:
        global_groups = df.groupby(topic_col)[text_col].apply(
            lambda x: " ".join(x)
        ).reset_index()
        global_groups.columns = ["topic", "text"]
        global_groups = global_groups.sort_values("topic").reset_index(drop=True)
        global_tf = vectorizer.transform(global_groups["text"])
        global_tfidf = global_tf.multiply(idf).toarray()
        # L1 normalise global representation (matches BERTopic)
        global_row_sums = global_tfidf.sum(axis=1, keepdims=True)
        global_row_sums[global_row_sums == 0] = 1
        global_tfidf = global_tfidf / global_row_sums
        global_topic_to_idx = {
            int(t): i for i, t in enumerate(global_groups["topic"])
        }
    
    # --- L1 normalise per-year matrix (before tuning, matches BERTopic) ---
    if global_tuning or evolution_tuning:
        row_sums = tfidf_matrix.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        tfidf_matrix = tfidf_matrix / row_sums
    
    # --- Build index: (year, topic) -> row index ---
    yt_to_idx = {}
    for idx in range(len(groups)):
        y = groups.iloc[idx]["year"]
        t = groups.iloc[idx]["topic"]
        yt_to_idx[(y, t)] = idx
    
    # --- Evolution tuning: average with t-1 ---
    if evolution_tuning:
        for yi in range(1, len(years)):
            curr_year = years[yi]
            prev_year = years[yi - 1]
            for topic in topics:
                curr_key = (curr_year, topic)
                prev_key = (prev_year, topic)
                if curr_key in yt_to_idx and prev_key in yt_to_idx:
                    ci = yt_to_idx[curr_key]
                    pi = yt_to_idx[prev_key]
                    tfidf_matrix[ci] = (tfidf_matrix[ci] + tfidf_matrix[pi]) / 2.0
    
    # --- Global tuning: average with global representation ---
    if global_tuning:
        for idx in range(len(groups)):
            topic = int(groups.iloc[idx]["topic"])
            if topic in global_topic_to_idx:
                gi = global_topic_to_idx[topic]
                tfidf_matrix[idx] = (tfidf_matrix[idx] + global_tfidf[gi]) / 2.0
    
    # --- Final L2 normalise before extracting top words ---
    row_norms = np.linalg.norm(tfidf_matrix, axis=1, keepdims=True)
    row_norms[row_norms == 0] = 1
    tfidf_matrix = tfidf_matrix / row_norms
    
    # Extract top words per (year, topic)
    topic_words_per_year = {}
    for idx in range(len(groups)):
        year = groups.iloc[idx]["year"]
        topic = groups.iloc[idx]["topic"]
        scores = tfidf_matrix[idx]
        top_indices = scores.argsort()[-top_n:][::-1]
        top_words = [vocab[i] for i in top_indices if scores[i] > 0]
        topic_words_per_year[(year, topic)] = top_words
    
    return topic_words_per_year


def calculate_coherence_for_words(
    topic_word_lists,
    texts_tokenized,
    dictionary,
) -> float:
    """Calculate C_v coherence given a list of topic word lists."""
    if len(topic_word_lists) == 0:
        return 0.0
    # Filter out empty or too-short topic word lists
    valid_topics = [tw for tw in topic_word_lists if len(tw) >= 2]
    if len(valid_topics) == 0:
        return 0.0
    
    cm = CoherenceModel(
        topics=valid_topics,
        texts=texts_tokenized,
        dictionary=dictionary,
        coherence='c_v',
        processes=1
    )
    return cm.get_coherence()


def rbo(list_1, list_2, p=0.9):
    """Rank-Biased Overlap between two ranked lists."""
    k = min(len(list_1), len(list_2))
    if k == 0:
        return 0.0
    rbo_score = 0.0
    for d in range(1, k + 1):
        set_1 = set(list_1[:d])
        set_2 = set(list_2[:d])
        agreement = len(set_1 & set_2) / d
        rbo_score += (p ** (d - 1)) * agreement
    rbo_score *= (1 - p)
    return rbo_score


def calculate_irbo(topics_words, p=0.9):
    """Calculate mean IRBO diversity across all topic pairs."""
    if len(topics_words) < 2:
        return 0.0
    irbo_scores = []
    for (i, j) in combinations(range(len(topics_words)), 2):
        similarity = rbo(topics_words[i], topics_words[j], p=p)
        irbo_scores.append(1.0 - similarity)
    return np.mean(irbo_scores)

## Load Models & Data

In [4]:
all_models = {}
all_data = {}
all_years = {}

for subject in LIST_SUBJECT:
    print(f"\nLoading {subject}...")
    
    # Load model
    model_path = MODEL_DIR / subject / "model"
    model = Top2Vec.load(str(model_path))
    all_models[subject] = model
    
    # Load data
    df = load_dataset(subject)
    df["topic"] = model.doc_top  # assign topic from model
    all_data[subject] = df
    
    years = sorted(df["year"].unique())
    all_years[subject] = years
    
    n_topics = model.get_num_topics()
    print(f"  {subject}: {len(df):,} docs, {n_topics} topics, "
          f"{len(years)} years ({years[0]}-{years[-1]})")
    print(f"  Topic sizes: min={model.topic_sizes.min()}, "
          f"max={model.topic_sizes.max()}, "
          f"mean={model.topic_sizes.mean():.0f}")

print(f"\n✅ All subjects loaded")


Loading cs...
  cs: 165,756 docs, 259 topics, 26 years (2000-2025)
  Topic sizes: min=85, max=4176, mean=640

Loading math...
  math: 157,085 docs, 209 topics, 26 years (2000-2025)
  Topic sizes: min=123, max=4528, mean=752

Loading physics...
  physics: 146,311 docs, 210 topics, 26 years (2000-2025)
  Topic sizes: min=143, max=3598, mean=697

✅ All subjects loaded


## Topic Prevalence Over Time

For each year, compute the proportion of documents belonging to each topic.
This shows how topics rise, fall, emerge, and disappear over time.

In [5]:
for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    years = all_years[subject]
    n_topics = model.get_num_topics()

    prevalence_csv = RESULT_DIR / subject / "topic_prevalence.csv"

    print(f"\n{'='*70}")
    print(f"Topic Prevalence: {subject.upper()} ({n_topics} topics)")
    print(f"{'='*70}")

    prevalence_rows = []

    for year in years:
        year_df = df[df["year"] == year]
        n_docs_year = len(year_df)
        topic_counts = year_df["topic"].value_counts()

        for topic_id in range(n_topics):
            count = topic_counts.get(topic_id, 0)
            proportion = count / n_docs_year if n_docs_year > 0 else 0.0

            # Get this topic's top words (from the global model)
            topic_words_arr, _, _ = model.get_topics(n_topics)
            top_words = ", ".join(topic_words_arr[topic_id][:5])

            prevalence_rows.append({
                "subject": subject,
                "year": year,
                "topic_id": topic_id,
                "doc_count": count,
                "total_docs_year": n_docs_year,
                "proportion": round(proportion, 6),
                "top_words": top_words,
            })

        # Summary for this year
        active_topics = (topic_counts > 0).sum()
        top_topic = topic_counts.idxmax()
        top_count = topic_counts.max()
        print(f"  {year}: {n_docs_year:,} docs, {active_topics}/{n_topics} active topics, "
              f"top=T{top_topic} ({top_count} docs)")

    prevalence_df = pd.DataFrame(prevalence_rows)
    prevalence_df.to_csv(prevalence_csv, index=False)
    print(f"\n  Saved to: {prevalence_csv} ({len(prevalence_df)} rows)")


Topic Prevalence: CS (259 topics)
  2000: 488 docs, 109/259 active topics, top=T1 (115 docs)
  2001: 594 docs, 111/259 active topics, top=T1 (69 docs)
  2002: 648 docs, 124/259 active topics, top=T1 (108 docs)
  2003: 825 docs, 125/259 active topics, top=T3 (91 docs)
  2004: 948 docs, 151/259 active topics, top=T1 (89 docs)
  2005: 1,000 docs, 154/259 active topics, top=T1 (81 docs)
  2006: 1,000 docs, 147/259 active topics, top=T1 (94 docs)
  2007: 1,000 docs, 144/259 active topics, top=T0 (63 docs)
  2008: 1,000 docs, 152/259 active topics, top=T1 (59 docs)
  2009: 1,000 docs, 160/259 active topics, top=T0 (64 docs)
  2010: 1,362 docs, 172/259 active topics, top=T0 (81 docs)
  2011: 1,622 docs, 184/259 active topics, top=T0 (112 docs)
  2012: 2,254 docs, 209/259 active topics, top=T0 (143 docs)
  2013: 2,719 docs, 219/259 active topics, top=T0 (167 docs)
  2014: 2,989 docs, 223/259 active topics, top=T0 (175 docs)
  2015: 3,345 docs, 242/259 active topics, top=T0 (173 docs)
  2016: 

## Topic Word Evolution (c-TF-IDF per Year)

Compute c-TF-IDF for each topic at each time point to see how
topic word compositions change over time. This is the core of
BERTopic's `topics_over_time()` approach.

In [6]:
all_topic_words_per_year = {}

for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    n_topics = model.get_num_topics()

    evolution_csv = RESULT_DIR / subject / "topic_word_evolution.csv"

    print(f"\n{'='*70}")
    print(f"Topic Word Evolution: {subject.upper()}")
    print(f"{'='*70}")

    start = time.time()
    topic_words_per_year = compute_ctfidf_per_year(
        df, topic_col="topic", text_col="text", top_n=TOP_N_WORDS,
        global_tuning=True, evolution_tuning=True,
    )
    elapsed = time.time() - start
    all_topic_words_per_year[subject] = topic_words_per_year

    print(f"  c-TF-IDF computed in {elapsed:.1f}s")
    print(f"  (year, topic) groups: {len(topic_words_per_year)}")

    # Save word evolution
    evolution_rows = []
    for (year, topic_id), words in sorted(topic_words_per_year.items()):
        evolution_rows.append({
            "subject": subject,
            "year": year,
            "topic_id": topic_id,
            "top_words": ", ".join(words),
        })

    evolution_df = pd.DataFrame(evolution_rows)
    evolution_df.to_csv(evolution_csv, index=False)
    print(f"  Saved to: {evolution_csv}")

    # Show example: topic 0 across a few years
    print(f"\n  Example — Topic 0 word evolution:")
    for year in [2000, 2005, 2010, 2015, 2020, 2025]:
        key = (year, 0)
        if key in topic_words_per_year:
            words = ", ".join(topic_words_per_year[key][:5])
            print(f"    {year}: {words}")


Topic Word Evolution: CS


  c-TF-IDF computed in 67.9s
  (year, topic) groups: 5195
  Saved to: ../../../../results/top2vec/temporal/cs/topic_word_evolution.csv

  Example — Topic 0 word evolution:
    2000: graphs, graph, vertex, rectilinear, polygon
    2005: graphs, graph, log, vertex, problem
    2010: graphs, graph, log, vertex, vertices
    2015: graphs, graph, log, vertex, gg
    2020: graphs, graph, log, gg, vertex
    2025: graphs, graph, log, vertex, gg

Topic Word Evolution: MATH
  c-TF-IDF computed in 29.2s
  (year, topic) groups: 5134
  Saved to: ../../../../results/top2vec/temporal/math/topic_word_evolution.csv

  Example — Topic 0 word evolution:
    2000: left, right, solutions, du, equation
    2005: equation, solutions, equations, wave, dinger
    2010: equation, solutions, mathbb, nonlinear, equations
    2015: solutions, equation, mathbb, frac, quad
    2020: equation, mathbb, solutions, frac, equations
    2025: mathbb, equation, solutions, frac, nabla

Topic Word Evolution: PHYSICS
  c-TF-

## Per-Year Coherence & IRBO

For each year, compute coherence and IRBO using that year's c-TF-IDF
topic word lists. This measures how well-defined and diverse the topics
are at each point in time.

In [7]:
for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    years = all_years[subject]
    n_topics = model.get_num_topics()
    topic_words_per_year = all_topic_words_per_year[subject]

    metrics_csv = RESULT_DIR / subject / "per_year_metrics.csv"

    print(f"\n{'='*70}")
    print(f"Per-Year Metrics: {subject.upper()} ({n_topics} topics)")
    print(f"{'='*70}")

    metrics_rows = []

    for year in years:
        year_df = df[df["year"] == year]
        n_docs = len(year_df)
        texts_tokenized = [text.split() for text in year_df["text"].tolist()]
        dictionary = Dictionary(texts_tokenized)

        # Get topic word lists for this year
        active_topics = sorted(year_df["topic"].unique())
        year_topic_words = []
        for tid in active_topics:
            key = (year, tid)
            if key in topic_words_per_year and len(topic_words_per_year[key]) >= 2:
                year_topic_words.append(topic_words_per_year[key])

        # Coherence
        coherence = calculate_coherence_for_words(
            year_topic_words, texts_tokenized, dictionary
        )

        # IRBO
        irbo_mean = calculate_irbo(year_topic_words, p=RBO_P)

        # Topic Quality = harmonic mean
        if coherence + irbo_mean > 0:
            topic_quality = 2 * coherence * irbo_mean / (coherence + irbo_mean)
        else:
            topic_quality = 0.0

        n_active = len(active_topics)

        print(f"  {year}: {n_docs:,} docs, {n_active} active topics | "
              f"Quality={topic_quality:.4f} (C={coherence:.4f}, IRBO={irbo_mean:.4f})")

        metrics_rows.append({
            "subject": subject,
            "year": year,
            "num_docs": n_docs,
            "num_topics_total": n_topics,
            "num_topics_active": n_active,
            "coherence_cv": round(coherence, 6),
            "irbo_mean": round(irbo_mean, 6),
            "topic_quality": round(topic_quality, 6),
        })

    metrics_df = pd.DataFrame(metrics_rows)
    metrics_df.to_csv(metrics_csv, index=False)
    print(f"\n  Saved to: {metrics_csv}")


Per-Year Metrics: CS (259 topics)
  2000: 488 docs, 109 active topics | Quality=0.7378 (C=0.5852, IRBO=0.9983)
  2001: 594 docs, 111 active topics | Quality=0.6616 (C=0.4950, IRBO=0.9973)
  2002: 648 docs, 124 active topics | Quality=0.6932 (C=0.5315, IRBO=0.9965)
  2003: 825 docs, 125 active topics | Quality=0.6258 (C=0.4561, IRBO=0.9967)
  2004: 948 docs, 151 active topics | Quality=0.6651 (C=0.4990, IRBO=0.9968)
  2005: 1,000 docs, 154 active topics | Quality=0.6466 (C=0.4784, IRBO=0.9971)
  2006: 1,000 docs, 147 active topics | Quality=0.6764 (C=0.5118, IRBO=0.9971)
  2007: 1,000 docs, 144 active topics | Quality=0.6785 (C=0.5142, IRBO=0.9969)
  2008: 1,000 docs, 152 active topics | Quality=0.6759 (C=0.5113, IRBO=0.9968)
  2009: 1,000 docs, 160 active topics | Quality=0.6500 (C=0.4822, IRBO=0.9969)
  2010: 1,362 docs, 172 active topics | Quality=0.6606 (C=0.4940, IRBO=0.9966)
  2011: 1,622 docs, 184 active topics | Quality=0.6543 (C=0.4871, IRBO=0.9962)
  2012: 2,254 docs, 209 act

## Topic Trends: Emerging, Growing, and Declining Topics

Identify which topics are trending up, trending down,
or stable over the full time period.

In [8]:
from scipy.stats import linregress

for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    # Get topic words from evolution CSV
    evo_path = RESULT_DIR / subject / "topic_word_evolution.csv"
    evo_df = pd.read_csv(evo_path)
    last_year = evo_df['year'].max()
    last_evo = evo_df[evo_df['year'] == last_year]
    global_tw = {}
    for _, erow in last_evo.iterrows():
        global_tw[int(erow['topic_id'])] = [w.strip() for w in str(erow['top_words']).split(',')][:5]

    rows = []
    topic_ids = sorted(df["topic"].unique())
    for tid in topic_ids:
        topic_df = df[df["topic"] == tid]
        if len(topic_df) == 0:
            continue

        year_counts = topic_df["year"].value_counts().sort_index()
        total_per_year = df["year"].value_counts().sort_index()
        proportions = (year_counts / total_per_year).fillna(0)

        # Align proportions with all years
        all_years = sorted(total_per_year.index)
        prop_aligned = proportions.reindex(all_years, fill_value=0.0)

        years_arr = np.array(all_years, dtype=float)
        props_arr = prop_aligned.values.astype(float)

        # Linear regression
        slope, intercept, r_val, p_val, std_err = linregress(years_arr, props_arr)

        # Early/late for display
        topic_years = sorted(year_counts.index)
        early_mean = proportions[topic_years[:5]].mean() if len(topic_years) >= 5 else proportions.mean()
        late_mean = proportions[topic_years[-5:]].mean() if len(topic_years) >= 5 else proportions.mean()

        # Classify by slope significance
        if p_val < 0.05 and slope > 0:
            trend_label = "GROWING"
        elif p_val < 0.05 and slope < 0:
            trend_label = "DECLINING"
        else:
            trend_label = "STABLE"

        top_words = global_tw.get(tid, ["?"])
        rows.append({
            "subject": subject, "topic_id": tid,
            "top_words": ", ".join(top_words),
            "total_docs": len(topic_df),
            "first_year": year_counts.index.min(),
            "last_year": year_counts.index.max(),
            "early_proportion": round(early_mean, 6),
            "late_proportion": round(late_mean, 6),
            "slope": round(slope, 8),
            "r_squared": round(r_val**2, 4),
            "p_value": round(p_val, 6),
            "trend": trend_label,
        })

    trends_df = pd.DataFrame(rows)
    trends_df.to_csv(RESULT_DIR / subject / "topic_trends.csv", index=False)

    g = len(trends_df[trends_df["trend"] == "GROWING"])
    s = len(trends_df[trends_df["trend"] == "STABLE"])
    d = len(trends_df[trends_df["trend"] == "DECLINING"])
    print(f"  {subject.upper()}: Growing={g}, Stable={s}, Declining={d}")


  CS: Growing=148, Stable=75, Declining=36
  MATH: Growing=86, Stable=69, Declining=54
  PHYSICS: Growing=93, Stable=64, Declining=53


## Top 5 Growing & Declining Topics

In [9]:
for subject in LIST_SUBJECT:
    trends_df = pd.read_csv(RESULT_DIR / subject / "topic_trends.csv")

    print(f"\n{'='*80}")
    print(f"  {subject.upper()}")
    print(f"{'='*80}")

    growing = trends_df[trends_df['trend'] == 'GROWING'].sort_values('slope', ascending=False)
    declining = trends_df[trends_df['trend'] == 'DECLINING'].sort_values('slope', ascending=True)

    print(f"\n  " + chr(0x1F4C8) + f" TOP 5 GROWING (steepest positive slope):")
    for _, row in growing.head(5).iterrows():
        print(f"    T{int(row['topic_id']):>3} | slope={row['slope']:+.6f} "
              f"R\u00B2={row['r_squared']:.3f} | "
              f"{row['early_proportion']:.4f} \u2192 {row['late_proportion']:.4f} | "
              f"{row['top_words']}")

    print(f"\n  " + chr(0x1F4C9) + f" TOP 5 DECLINING (steepest negative slope):")
    for _, row in declining.head(5).iterrows():
        print(f"    T{int(row['topic_id']):>3} | slope={row['slope']:+.6f} "
              f"R\u00B2={row['r_squared']:.3f} | "
              f"{row['early_proportion']:.4f} \u2192 {row['late_proportion']:.4f} | "
              f"{row['top_words']}")



  CS

  📈 TOP 5 GROWING (steepest positive slope):
    T  2 | slope=+0.000886 R²=0.609 | 0.0070 → 0.0236 | speech, audio, speaker, asr, music
    T  4 | slope=+0.000780 R²=0.661 | 0.0051 → 0.0191 | learning, rl, reinforcement, policy, reward
    T  5 | slope=+0.000714 R²=0.621 | 0.0017 → 0.0162 | visual, language, image, multimodal, models
    T 10 | slope=+0.000666 R²=0.746 | 0.0030 → 0.0119 | segmentation, object, detection, semantic, image
    T  9 | slope=+0.000633 R²=0.789 | 0.0013 → 0.0135 | graph, gnns, graphs, node, learning

  📉 TOP 5 DECLINING (steepest negative slope):
    T  1 | slope=-0.005874 R²=0.726 | 0.1431 → 0.0123 | logic, semantics, logics, programs, calculus
    T  0 | slope=-0.001744 R²=0.535 | 0.0522 → 0.0161 | graphs, graph, log, vertex, gg
    T  3 | slope=-0.001629 R²=0.363 | 0.0560 → 0.0131 | memory, performance, gpu, data, hardware
    T  6 | slope=-0.001371 R²=0.542 | 0.0419 → 0.0122 | quantum, classical, qubit, circuits, qubits
    T 24 | slope=-0.001279 

## Evolution Summary

In [10]:
for subject in LIST_SUBJECT:
    metrics_csv = RESULT_DIR / subject / "per_year_metrics.csv"
    trends_csv = RESULT_DIR / subject / "topic_trends.csv"

    if not metrics_csv.exists():
        print(f"{subject}: missing metrics, skipping")
        continue

    metrics_df = pd.read_csv(metrics_csv)
    trends_df = pd.read_csv(trends_csv)

    model = all_models[subject]
    n_topics = model.get_num_topics()

    growing = len(trends_df[trends_df["trend"] == "GROWING"])
    declining = len(trends_df[trends_df["trend"] == "DECLINING"])
    stable = len(trends_df[trends_df["trend"] == "STABLE"])

    summary_data = {
        "subject": subject,
        "num_topics": n_topics,
        "num_years": len(metrics_df),
        "coherence_mean": round(metrics_df["coherence_cv"].mean(), 6),
        "coherence_std": round(metrics_df["coherence_cv"].std(), 6),
        "irbo_mean": round(metrics_df["irbo_mean"].mean(), 6),
        "irbo_std": round(metrics_df["irbo_mean"].std(), 6),
        "quality_mean": round(metrics_df["topic_quality"].mean(), 6),
        "quality_std": round(metrics_df["topic_quality"].std(), 6),
        "topics_growing": growing,
        "topics_stable": stable,
        "topics_declining": declining,
    }
    summary_df = pd.DataFrame([summary_data])
    summary_csv = RESULT_DIR / subject / "evolution_summary.csv"
    summary_df.to_csv(summary_csv, index=False)
    print(f"\n  {subject.upper()} summary saved to: {summary_csv}")


  CS summary saved to: ../../../../results/top2vec/temporal/cs/evolution_summary.csv

  MATH summary saved to: ../../../../results/top2vec/temporal/math/evolution_summary.csv

  PHYSICS summary saved to: ../../../../results/top2vec/temporal/physics/evolution_summary.csv


## Final Results

In [11]:
print("\n" + "=" * 110)
print("TOP2VEC TEMPORAL ANALYSIS: FINAL RESULTS")
print("=" * 110)

for subject in LIST_SUBJECT:
    metrics_csv = RESULT_DIR / subject / "per_year_metrics.csv"
    trends_csv = RESULT_DIR / subject / "topic_trends.csv"

    if not metrics_csv.exists():
        print(f"\n{subject.upper()}: No results found")
        continue

    metrics_df = pd.read_csv(metrics_csv)
    trends_df = pd.read_csv(trends_csv)

    model = all_models[subject]
    n_topics = model.get_num_topics()

    growing = len(trends_df[trends_df["trend"] == "GROWING"])
    declining = len(trends_df[trends_df["trend"] == "DECLINING"])
    stable = len(trends_df[trends_df["trend"] == "STABLE"])

    sep = chr(9472)
    print(f"\n{sep*60}")
    print(f"  Subject:        {subject.upper()}")
    print(f"  Num topics:     {n_topics}")
    print(f"  Years:          {len(metrics_df)}")
    print(f"  Coherence:      {metrics_df['coherence_cv'].mean():.4f} +/- {metrics_df['coherence_cv'].std():.4f}")
    print(f"  IRBO:           {metrics_df['irbo_mean'].mean():.4f} +/- {metrics_df['irbo_mean'].std():.4f}")
    print(f"  Topic Quality:  {metrics_df['topic_quality'].mean():.4f} +/- {metrics_df['topic_quality'].std():.4f}")
    print(f"  Trends:         ↑{growing} growing, →{stable} stable, ↓{declining} declining")
    print(f"{sep*60}")

print("\n" + "=" * 110)
print("Per-Year Details:")
print("=" * 110)

for subject in LIST_SUBJECT:
    metrics_csv = RESULT_DIR / subject / "per_year_metrics.csv"
    if not metrics_csv.exists():
        continue

    metrics_df = pd.read_csv(metrics_csv)

    print(f"\n{subject.upper()}:")
    print(metrics_df[["year", "num_docs", "num_topics_active",
                     "coherence_cv", "irbo_mean", "topic_quality"]].to_string(index=False))
    print()


TOP2VEC TEMPORAL ANALYSIS: FINAL RESULTS

────────────────────────────────────────────────────────────
  Subject:        CS
  Num topics:     259
  Years:          26
  Coherence:      0.5321 +/- 0.0574
  IRBO:           0.9938 +/- 0.0036
  Topic Quality:  0.6912 +/- 0.0470
  Trends:         ↑148 growing, →75 stable, ↓36 declining
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  Subject:        MATH
  Num topics:     209
  Years:          26
  Coherence:      0.5233 +/- 0.0637
  IRBO:           0.9918 +/- 0.0035
  Topic Quality:  0.6828 +/- 0.0537
  Trends:         ↑86 growing, →69 stable, ↓54 declining
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  Subject:        PHYSICS
  Num topics:     210
  Years:          26
  Coherence:      0.5518 +/- 0.0681
  IRBO:           0.9943 +/- 0.0014
  Topic Quality:  0.7073 +/- 0.0557
  Trends:   